# Bridge Dashboard Generator

This notebook generates an interactive HTML dashboard for bridge engineers by aggregating data from multiple sources:

- **TIMs** - Transportation Information Management System
- **Assetwise** - Asset management data
- **BrR** - Bridge Reporting system
- **Historic Resources** - Historical bridge information

The dashboard provides comprehensive summaries and visualizations to support bridge inspection and maintenance decision-making.

In [69]:
# User Input Parameters
district = 6  # e.g., "1", "2", "3", etc.
county = "FRA"  # e.g., "Franklin", "Cuyahoga", etc.
route = 71  # e.g., "I-71", "US-23", "SR-315", etc.

# Gathering Data

Start with TIMs since it's the easiest to query and filter results from,

In [70]:
from src.civilpy.state.ohio.DOT.TIMS import get_bridge_sfns_by_district

In [71]:
# Define the endpoint and filter

district = 6  # Update the district here to change the bridges listed

# Call the function
d6_bridges = get_bridge_sfns_by_district(district=6)

Querying API for District 06...
URL: https://tims.dot.state.oh.us/ags/rest/services/Assets/Bridge_Inventory/MapServer/0/query
Filter: DISTRICT = '06'

Success! Found 4632 features.


In [72]:
# Sample 5 of the returned bridges to make sure we got SFNs
d6_bridges[-5:]

['8035725', '8036748', '8037817', '8060018', '8060398']

# Gather Assetwise Data

[Endpoint Documentation](https://ohiodot-it-api.bentley.com/swagger/index.html)

In [73]:
from civilpy.state.ohio.DOT.AssetWise import get_assetwise_secrets

In [74]:
username, password = get_assetwise_secrets()

In [75]:
import requests
from requests.auth import HTTPBasicAuth

def get_bridge_by_sfn(
    sfn: str,
    include_coordinates: bool = True,
    include_parent: bool = False
) -> dict:
    """
    Retrieves a specific bridge asset by its SFN (Structure File Number) using the AssetWise API.

    Args:
        sfn (str): The SFN/asset code of the bridge to retrieve (e.g., '0702870').
        include_coordinates (bool): Whether to include asset coordinates in the response. Defaults to True.
        include_parent (bool): Whether to include the parent as_id in the response. Defaults to False.

    Returns:
        dict: A dictionary representing the bridge asset data.

    Raises:
        requests.exceptions.HTTPError: If the API request encounters an HTTP error.
    """
    base_url = "https://ohiodot-it-api.bentley.com"
    api_url = f"{base_url}/api/Asset/GetAssetByAsCode/{sfn}"

    query_params = {
        "IncludeCoordinates": include_coordinates,
        "IncludeParent": include_parent
    }

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        api_url,
        params=query_params,
        headers=headers,
        auth=HTTPBasicAuth(username, password)
    )

    response.raise_for_status()
    return response.json()['data']

In [76]:
def get_bridge_general_data(asset_id: int) -> dict:
    """
    Retrieves detailed element-level field data for a specific bridge asset using the AssetWise API.

    This returns all the inspection and condition data fields for the bridge, including
    element-level inspection data, ratings, and other detailed information.

    Args:
        asset_id (int): The unique ID of the bridge asset (as_id from get_bridge_by_sfn).

    Returns:
        dict: A dictionary containing the element-level field data for the bridge.

    Raises:
        requests.exceptions.HTTPError: If the API request encounters an HTTP error.
    """
    base_url = "https://ohiodot-it-api.bentley.com"
    api_url = f"{base_url}/api/AssetDetailField/GetFieldsByAsId/{asset_id}"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        api_url,
        headers=headers,
        auth=HTTPBasicAuth(username, password)
    )

    response.raise_for_status()
    return response.json()

In [77]:
# Get inspection reports which contain element-level condition data
def get_bridge_inspection_reports(asset_id: int) -> list:
    """
    Retrieves all inspection reports for a specific bridge asset.

    Inspection reports contain element-level condition data, ratings, and defects.

    Args:
        asset_id (int): The unique ID of the bridge asset (as_id from get_bridge_by_sfn).

    Returns:
        list: A list of inspection report dictionaries for the bridge.

    Raises:
        requests.exceptions.HTTPError: If the API request encounters an HTTP error.
    """
    base_url = "https://ohiodot-it-api.bentley.com"
    api_url = f"{base_url}/api/AssetTask/GetAssetTasksByAsId/{asset_id}"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        api_url,
        headers=headers,
        auth=HTTPBasicAuth(username, password)
    )

    response.raise_for_status()
    return response.json().get('data', [])

In [78]:
def get_report_element_data(report_guid: str) -> dict:
    """
    Retrieves element-level inspection data for a specific inspection report.

    This gets the actual condition ratings and defects for bridge elements
    (deck, superstructure, substructure, etc.) from a completed inspection.

    Args:
        report_guid (str): The GUID of the inspection report (ast_guid from get_bridge_inspection_reports).

    Returns:
        dict: Element-level inspection data including condition states and ratings.

    Raises:
        requests.exceptions.HTTPError: If the API request encounters an HTTP error.
    """
    base_url = "https://ohiodot-it-api.bentley.com"
    api_url = f"{base_url}/api/Value/GetValuesByAstGuid/{report_guid}"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        api_url,
        headers=headers,
        auth=HTTPBasicAuth(username, password)
    )

    response.raise_for_status()
    return response.json()

In [79]:
# First get the bridge asset data using the SFN
bridge_asset = get_bridge_by_sfn(d6_bridges[0])

# Then get the general data using the asset ID
assetwise_general_data = get_bridge_general_data(bridge_asset['as_id'])

In [80]:
reports = get_bridge_inspection_reports(bridge_asset['as_id'])

HTTPError: 404 Client Error: Not Found for url: https://ohiodot-it-api.bentley.com/api/AssetTask/GetAssetTasksByAsId/78580

In [ ]:
if reports:
    latest_report = reports[0]  # Assuming sorted by date, may need to sort
    element_data = get_report_element_data(latest_report['ast_guid'])
    print(element_data)

In [ ]:
# Get all inspection reports for this bridge
inspection_reports = get_bridge_inspection_reports(bridge_asset['as_id'])

# Display how many reports were found
print(f"Found {len(inspection_reports)} inspection reports for bridge {d6_bridges[0]}")

# Show the most recent report (if any exist)
if inspection_reports:
    latest_report = inspection_reports[0]
    print(f"\nLatest report GUID: {latest_report.get('ast_guid')}")
    print(f"Report Name: {latest_report.get('ast_name')}")
    print(f"Report Date: {latest_report.get('ast_completed_date')}")
else:
    print("No inspection reports found")

In [ ]:
# Get element-level inspection data from the latest report
if inspection_reports:
    latest_report_guid = inspection_reports[0]['ast_guid']
    element_inspection_data = get_report_element_data(latest_report_guid)

    print(f"Retrieved {element_inspection_data.get('totalCount', 0)} element data fields")

    # Display the element data structure
    element_inspection_data
else:
    print("No reports available to retrieve element data from")